In [49]:
import numpy as np
import heapq

# Problem 4

In [90]:
def system_A(c=3, lam=6, mu=3, T=10):

    state_times = [[] for _ in range(c)]

    for queue in range(c):
        t = 0
        Q_t = 0
        events = []

        heapq.heappush(events, (np.random.exponential(1/(lam/c)), 'arrival'))

        while events and events[0][0] < T:
            t, event_type = heapq.heappop(events)

            state_times[queue].append((t, Q_t))

            if event_type == 'arrival':
                if Q_t == 0:
                    heapq.heappush(events, (t + np.random.exponential(1/mu), 'departure'))
                Q_t += 1
                heapq.heappush(events, (t + np.random.exponential(1/(lam/c)), 'arrival'))
            elif event_type == 'departure':
                Q_t -= 1
                if Q_t >= 1:
                    heapq.heappush(events, (t + np.random.exponential(1/mu), 'departure'))




    all_state_times = []
    for queue_idx in range(len(state_times)): # loop through each queue
        for t, q in state_times[queue_idx]: # for each tuple
            all_state_times.append((t, queue_idx, q)) # add the tuple to combined list

    all_state_times.sort(key=lambda x: x[0])  # Sort combined list by time

    Q_current = [0] * c
    combined_state_times = []

    for t, queue_idx, q in all_state_times: # loop through combined list (now sorted by time)
        Q_current[queue_idx] = q # some queue has changed, update the current state of that queue
        Q_total = sum(Q_current) # take new sum
        combined_state_times.append((t, Q_total)) # add to final list


    return combined_state_times


def system_B(c=3, lam=6, mu=3, T=10):

    t = 0
    Q_t = 0
    events = []
    state_times = []

    heapq.heappush(events, (np.random.exponential(1/lam), 'arrival'))

    while events and events[0][0] < T:
        t, event_type = heapq.heappop(events)

        state_times.append((t, Q_t))

        if event_type == 'arrival':
            if Q_t == 0: # If the system is empty, schedule a departure for the new customer (served immediately)
                heapq.heappush(events, (t + np.random.exponential(1/(c*mu)), 'departure'))
            Q_t += 1
            heapq.heappush(events, (t + np.random.exponential(1/lam), 'arrival')) # Schedule the next arrival
        elif event_type == 'departure':
            Q_t -= 1
            if Q_t >= 1: # If there are still customers in the system, schedule the next departure (next customer enters service)
                heapq.heappush(events, (t + np.random.exponential(1/(c*mu)), 'departure'))

    return state_times

def system_C(c=3, lam=6, mu=3, T=10):

    t = 0
    Q_t = 0
    events = []
    state_times = []

    heapq.heappush(events, (np.random.exponential(1/lam), 'arrival'))

    while events and events[0][0] < T:
        t, event_type = heapq.heappop(events)

        state_times.append((t, Q_t))

        if event_type == 'arrival':
            if Q_t < c: # If customer can be served immediately
                heapq.heappush(events, (t + np.random.exponential(1/mu), 'departure'))
            Q_t += 1
            heapq.heappush(events, (t + np.random.exponential(1/lam), 'arrival'))
        elif event_type == 'departure':
            Q_t -= 1
            if Q_t >= c: # If there are more customers than servers, schedule the next departure (next customer enters service)
                heapq.heappush(events, (t + np.random.exponential(1/mu), 'departure'))

    return state_times

In [91]:
def estimate_distributions(state_times, T):
    time_in_state = {}

    for i in range(len(state_times) - 1):
        t, Q = state_times[i]
        next_t, _ = state_times[i + 1]

        time = next_t - t

        if Q not in time_in_state:
            time_in_state[Q] = 0
        time_in_state[Q] += time

    last_t, last_Q = state_times[-1]
    if last_Q not in time_in_state:
        time_in_state[last_Q] = 0
    time_in_state[last_Q] += T - last_t

    total_time = sum(time_in_state.values())
    distribution = {Q: time / total_time for Q, time in time_in_state.items()}

    return distribution

a = system_A()
estimate_distributions_A = estimate_distributions(a, 10)
print("Estimated distribution for System A:", estimate_distributions_A)

b = system_B()
estimate_distributions_B = estimate_distributions(b, 10)
print("Estimated distribution for System B:", estimate_distributions_B)

c = system_C()
estimate_distributions_C = estimate_distributions(c, 10)
print("Estimated distribution for System C:", estimate_distributions_C)

Estimated distribution for System A: {0: 0.012500724070595833, 1: 0.04783481811663796, 2: 0.09612476320131164, 3: 0.0878082293642619, 4: 0.037859066559547835, 5: 0.10069249929364081, 6: 0.060166278841068295, 7: 0.1025779141294646, 8: 0.06742407273851159, 9: 0.07231919988230905, 10: 0.21614403088749634, 11: 0.0798254560573108, 12: 0.018722946857843436}
Estimated distribution for System B: {0: 0.1335084376885124, 1: 0.418397901769608, 2: 0.17477118488224022, 3: 0.13325720810012737, 4: 0.08372514848103736, 5: 0.014107494926335833, 6: 0.02024162829956372, 7: 0.019916768935892084, 8: 0.002074226916682774}
Estimated distribution for System C: {0: 0.11585605128767738, 1: 0.33027642525247236, 2: 0.18358487498411413, 3: 0.11221864850018226, 4: 0.0725917098792399, 5: 0.08865460986852135, 6: 0.09379164833061034, 7: 0.0030260318971822137}
